<a href="https://colab.research.google.com/github/ashivashankars/CMPE258_Assignments/blob/main/11_text_nlp_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 11 — Text / NLP Augmentation with nlpaug

## What This Notebook Covers
Text augmentation is fundamentally different from image augmentation.
Images exist in a continuous space — you can smoothly interpolate pixel
values. Text exists in a discrete space — swapping one word for another
can completely change the meaning.

Good text augmentation must:
- Preserve the semantic meaning (label should not change)
- Produce grammatically plausible sentences
- Be label-preserving (a positive review stays positive)

**Libraries used:**
- `nlpaug` — the most comprehensive NLP augmentation library
- `transformers` (HuggingFace) — for contextual word embeddings

**Techniques covered:**
- Character-level: keyboard typos, OCR errors, random insert/delete/swap
- Word-level: synonym replacement (WordNet), random swap, random delete
- Embedding-level: replace with similar word vectors (word2vec/GloVe)
- Contextual: BERT-based insertion and substitution
- Sentence-level: back-translation
- Composition: chaining multiple augmenters
- Text classifier: sentiment analysis with and without augmentation
- A/B test: baseline vs augmented training set

**Dataset:** IMDb sentiment (positive / negative reviews)


In [ ]:
%pip uninstall -y transformers tokenizers
%pip install -U pip setuptools wheel
%pip install nlpaug transformers tokenizers


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.9 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]


In [ ]:
import nlpaug.model.lang_models.bert as bert_mod

def token2id_fix(self, token):
    return self.tokenizer.convert_tokens_to_ids(token)

bert_mod.Bert.token2id = token2id_fix

After restarting the runtime, you will need to re-run all cells from the beginning to ensure the correct `transformers` version is used. Specifically, re-run the initial `pip install` cell (`diof-vYTyj8F`), then continue from `SmBiWuIUyj8G` onwards.

In [ ]:
!pip install nlpaug transformers datasets --quiet

import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.flow as nafl
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random
import re

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('nlpaug imported successfully')
print('Device:', device)

nlpaug imported successfully
Device: cuda


## 1. Load IMDb Dataset


In [ ]:
from datasets import load_dataset

print('Loading IMDb dataset...')
imdb = load_dataset('imdb')

# Use a small subset for speed — augmentation is slow (especially BERT-based)
N_TRAIN = 2000
N_TEST  = 500

train_texts  = imdb['train']['text'][:N_TRAIN]
train_labels = imdb['train']['label'][:N_TRAIN]
test_texts   = imdb['test']['text'][:N_TEST]
test_labels  = imdb['test']['label'][:N_TEST]

# Truncate long reviews to 256 words for speed
def truncate(text, max_words=256):
    return ' '.join(text.split()[:max_words])

train_texts = [truncate(t) for t in train_texts]
test_texts  = [truncate(t) for t in test_texts]

print(f'Train: {len(train_texts)} reviews')
print(f'Test : {len(test_texts)} reviews')
print(f'Label distribution: {Counter(train_labels)}')
print()
print('Sample review (positive):')
print(train_texts[0][:300], '...')

Loading IMDb dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train: 2000 reviews
Test : 500 reviews
Label distribution: Counter({0: 2000})

Sample review (positive):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...


## 2. Visualisation Helper


In [ ]:
def show_augmentation(augmenter, texts, n=3, title=''):
    """
    Apply augmenter to n texts and print original vs augmented side by side.
    Highlights changed words by comparing tokens.
    """
    print(f'=== {title} ===')
    print()
    for i, text in enumerate(texts[:n]):
        augmented = augmenter.augment(text)
        if isinstance(augmented, list):
            augmented = augmented[0]

        print(f'--- Example {i+1} ---')
        print(f'ORIGINAL : {text[:200]}')
        print(f'AUGMENTED: {augmented[:200]}')

        # Count changed tokens
        orig_tokens = text.split()
        aug_tokens  = augmented.split()
        n_changed = sum(1 for a, b in zip(orig_tokens, aug_tokens) if a != b)
        print(f'Changes  : {n_changed} tokens modified')
        print()
    print('-' * 60)
    print()

# Sample texts for demos
DEMO_TEXTS = [
    "This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.",
    "I was really disappointed with this film. The plot was boring and the characters were poorly developed.",
    "An incredible cinematic experience with stunning visuals and a powerful emotional journey."
]

print('Demo texts ready.')

Demo texts ready.


## 3. Character-Level Augmentation

Character-level augmentation simulates typos, OCR errors, and keyboard
mistakes. It creates robustness to noisy input — common in social media,
user reviews, and speech-to-text output.

**Key concept:** `aug_char_p` controls what fraction of characters are
modified. Keep it low (0.1-0.2) to preserve readability.


In [ ]:
# Keyboard augmenter: replaces characters with adjacent keys on QWERTY keyboard
# Simulates typing errors — 'the' might become 'tge' or 'tje'
keyboard_aug = nac.KeyboardAug(
    aug_char_p=0.1,     # modify 10% of characters
    aug_word_p=0.1,     # apply to 10% of words
    include_special_char=False,
    include_numeric=False
)
show_augmentation(keyboard_aug, DEMO_TEXTS, title='KeyboardAug — QWERTY typos')

=== KeyboardAug — QWERTY typos ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: ThOs mLvie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
Changes  : 2 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I was really dksappoinFed with this film. The plot was boring and the characters Dere poorly developed.
Changes  : 2 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: An incredible cinematic experience qith stunning visuals and a powerful emotional jlurney.
Changes  : 2 tokens modified

------------------------------------------------------------



In [ ]:
# OCR augmenter: replaces characters with visually similar ones
# '0' <-> 'O', '1' <-> 'l', 'rn' <-> 'm'
# Simulates errors from optical character recognition systems
ocr_aug = nac.OcrAug(
    aug_char_p=0.1,
    aug_word_p=0.15
)
show_augmentation(ocr_aug, DEMO_TEXTS, title='OcrAug — visually similar character substitutions')

=== OcrAug — visually similar character substitutions ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: This movie was absolutely fantastic. The actin9 wa8 superb and the story kept me engaged throughout.
Changes  : 2 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I was really disappointed with this fi1m. The plot was boring and the characters were poorly developed.
Changes  : 1 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: An inckedi6le cinematic experience with stunning vi8ua1s and a powerful emotional journey.
Changes  : 2 tokens modified

------------------------------------------------------------



In [ ]:
# Random character augmenter: insert, delete, swap, or substitute random characters
# action='swap': swaps adjacent characters — common typing mistake
char_swap_aug = nac.RandomCharAug(
    action='swap',
    aug_char_p=0.1,
    aug_word_p=0.15
)
show_augmentation(char_swap_aug, DEMO_TEXTS, title='RandomCharAug (swap) — adjacent character transposition')

=== RandomCharAug (swap) — adjacent character transposition ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: This movei was absolutely fantastic. The actnig was superb and the tsory kept me engaged throughout.
Changes  : 3 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I was really dsiappionted with thsi flim. The plot was boring and the characters were poorly developed.
Changes  : 3 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: An incredible cinematic experience wtih stunning visuals and a powerful emotional juorney.
Changes  : 2 tokens modified

------------------------------------------------------------



## 4. Word-Level Augmentation

Word-level augmentation operates on whole tokens. These techniques
produce more semantically coherent augmented text than character-level,
because they replace/remove/swap entire words rather than individual characters.


In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

# Synonym replacement using WordNet
# Replaces words with synonyms from the WordNet lexical database
# 'fantastic' might become 'tremendous' or 'marvelous'
# aug_p controls what fraction of words are replaced
synonym_aug = naw.SynonymAug(
    aug_src='wordnet',
    aug_p=0.2           # replace 20% of eligible words
)
show_augmentation(synonym_aug, DEMO_TEXTS, title='SynonymAug (WordNet) — replace words with synonyms')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


=== SynonymAug (WordNet) — replace words with synonyms ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: This motion picture show was absolutely wild. The acting was superb and the story kept me engage throughout.
Changes  : 15 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I equal really foiled with this picture show. The plot was boring and the characters were poorly developed.
Changes  : 13 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: An incredible cinematic experience with sensational visuals and a brawny emotional journey.
Changes  : 2 tokens modified

------------------------------------------------------------



In [ ]:
# Random word deletion
# Randomly removes words from the sentence
# The model must learn to handle incomplete input
# Keep aug_p low (0.1-0.2) to avoid destroying sentence meaning
delete_aug = naw.RandomWordAug(
    action='delete',
    aug_p=0.15
)
show_augmentation(delete_aug, DEMO_TEXTS, title='RandomWordAug (delete) — randomly remove words')

=== RandomWordAug (delete) — randomly remove words ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: This movie was absolutely fantastic. The acting was the kept me engaged throughout.
Changes  : 5 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I was really disappointed with film. The plot was boring and were poorly developed.
Changes  : 9 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: An incredible with stunning visuals and a powerful emotional journey.
Changes  : 8 tokens modified

------------------------------------------------------------



In [ ]:
# Random word swap
# Swaps the positions of two random words in the sentence
# Sentence structure changes but all words remain
swap_aug = naw.RandomWordAug(
    action='swap',
    aug_p=0.15
)
show_augmentation(swap_aug, DEMO_TEXTS, title='RandomWordAug (swap) — swap word positions')

=== RandomWordAug (swap) — swap word positions ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: Movie this was absolutely fantastic. The acting was superb and the kept story me throughout engaged.
Changes  : 6 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I was really disappointed this with film. The plot boring was and the were characters poorly developed.
Changes  : 6 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: An incredible cinematic experience with stunning and visuals a emotional powerful journey.
Changes  : 4 tokens modified

------------------------------------------------------------



In [ ]:
# Random word insertion
# Inserts random words from the vocabulary at random positions
# Less common than deletion/swap — can introduce noise
insert_aug = naw.RandomWordAug(
    action='crop',      # 'crop' keeps a random contiguous span of words
    aug_p=0.1
)
show_augmentation(insert_aug, DEMO_TEXTS, title='RandomWordAug (crop) — keep random contiguous span')

=== RandomWordAug (crop) — keep random contiguous span ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: This movie was absolutely fantastic. The superb and the story kept me engaged throughout.
Changes  : 8 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I was really this film. The plot was boring and the characters were poorly developed.
Changes  : 12 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: An incredible cinematic experience with stunning a powerful emotional journey.
Changes  : 4 tokens modified

------------------------------------------------------------



## 5. Spelling Error Augmentation

Introduces realistic spelling mistakes based on a spelling error dictionary.
Common words like 'because' -> 'becuase', 'definitely' -> 'definately'.
Important for models that process user-generated content.


In [ ]:
# SpellingAug uses a dictionary of common spelling mistakes
spelling_aug = naw.SpellingAug(
    aug_p=0.2
)
show_augmentation(spelling_aug, DEMO_TEXTS, title='SpellingAug — common spelling mistakes')

=== SpellingAug — common spelling mistakes ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: Tis movie wad absolutely fantastic. The acting was superburb and the story keeped me engaged throughout.
Changes  : 4 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: I was really disappointed winth this film. The plot was boring ahd the characters werw poorly develped.
Changes  : 4 tokens modified

--- Example 3 ---
ORIGINAL : An incredible cinematic experience with stunning visuals and a powerful emotional journey.
AUGMENTED: And incredible cinematic expenience with stunning visuals cndy a powerful emotional journey.
Changes  : 3 tokens modified

------------------------------------------------------------



## 6. Contextual Word Augmentation (BERT-based)

BERT-based augmentation uses a masked language model to suggest
contextually appropriate replacements. Unlike WordNet synonyms,
BERT understands the surrounding context so suggestions are
grammatically and semantically coherent.

This is slower than other methods (requires a transformer forward pass)
but produces the highest quality augmented text.

**action='substitute'**: mask a word and let BERT predict what fits
**action='insert'**: ask BERT to insert a new contextually appropriate word


In [ ]:
print('Loading BERT augmenter (downloads ~440MB distilbert model)...')
print('This cell may take 2-3 minutes on first run.')

# ContextualWordEmbsAug uses a masked language model (distilbert by default)
# action='substitute': masks words and uses BERT to predict replacements
bert_sub_aug = naw.ContextualWordEmbsAug(
    model_path='distilbert-base-uncased',
    action='substitute',
    aug_p=0.15,         # replace 15% of words
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

show_augmentation(
    bert_sub_aug, DEMO_TEXTS, n=2,
    title='ContextualWordEmbsAug (BERT substitute) — context-aware replacement'
)

Loading BERT augmenter (downloads ~440MB distilbert model)...
This cell may take 2-3 minutes on first run.


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

[transformers] The following layers were not sharded: distilbert.embeddings.position_embeddings.weight, distilbert.embeddings.LayerNorm.bias, distilbert.transformer.layer.*.attention.q_lin.bias, distilbert.transformer.layer.*.attention.v_lin.weight, vocab_layer_norm.weight, distilbert.transformer.layer.*.ffn.lin1.bias, distilbert.embeddings.LayerNorm.weight, vocab_projector.weight, distilbert.transformer.layer.*.sa_layer_norm.weight, distilbert.transformer.layer.*.attention.q_lin.weight, distilbert.transformer.layer.*.output_layer_norm.bias, distilbert.transformer.layer.*.sa_layer_norm.bias, vocab_transform.weight, distilbert.transformer.layer.*.ffn.lin2.bias, distilbert.transformer.layer.*.attention.out_lin.weight, distilbert.embeddings.word_embeddings.weight, distilbert.transformer.layer.*.ffn.lin1.weight, distilbert.transformer.layer.*.attention.k_lin.weight, vocab_transform.bias, distilbert.transformer.layer.*.ffn.lin2.weight, distilbert.transformer.layer.*.attention.out_lin.bias, 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

=== ContextualWordEmbsAug (BERT substitute) — context-aware replacement ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: this movie was totally fantastic. my acting was superb and the story kept me hooked throughout.
Changes  : 4 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: i is always disappointed with this film. that plot was boring and the characters were poorly developed.
Changes  : 4 tokens modified

------------------------------------------------------------



In [ ]:
# action='insert': insert new words suggested by BERT at random positions
bert_ins_aug = naw.ContextualWordEmbsAug(
    model_path='distilbert-base-uncased',
    action='insert',
    aug_p=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

show_augmentation(
    bert_ins_aug, DEMO_TEXTS, n=2,
    title='ContextualWordEmbsAug (BERT insert) — context-aware insertion'
)

=== ContextualWordEmbsAug (BERT insert) — context-aware insertion ===

--- Example 1 ---
ORIGINAL : This movie was absolutely fantastic. The acting was superb and the story kept me engaged throughout.
AUGMENTED: this movie movie was absolutely not fantastic. the acting was superb and the story kept me engaged throughout.
Changes  : 15 tokens modified

--- Example 2 ---
ORIGINAL : I was really disappointed with this film. The plot was boring and the characters were poorly developed.
AUGMENTED: i was really disappointed with thinking this film. the comic plot was boring and the characters were poorly developed.
Changes  : 13 tokens modified

------------------------------------------------------------



## 7. Back-Translation

Back-translation translates text to another language then back to English.
The result is a paraphrase with the same meaning but different wording.
This is one of the strongest text augmentation techniques.

Pipeline: English -> French -> English

We demonstrate with HuggingFace translation models.
Note: this is slow and requires downloading translation models.


In [ ]:
from transformers import pipeline

print('Loading translation pipelines...')
en_to_fr = pipeline('translation', model='Helsinki-NLP/opus-mt-en-fr',
                     device=0 if torch.cuda.is_available() else -1)
fr_to_en = pipeline('translation', model='Helsinki-NLP/opus-mt-fr-en',
                     device=0 if torch.cuda.is_available() else -1)
print('Translation pipelines ready.')


def back_translate(text, max_length=512):
    """
    Translate English -> French -> English.
    Returns a paraphrase of the original text.
    """
    # Step 1: English -> French
    fr_result = en_to_fr(text, max_length=max_length)
    fr_text   = fr_result[0]['translation_text']

    # Step 2: French -> English
    en_result = fr_to_en(fr_text, max_length=max_length)
    en_text   = en_result[0]['translation_text']

    return en_text


# Demo back-translation
print('=== Back-Translation Demo (En -> Fr -> En) ===')
print()
for text in DEMO_TEXTS[:2]:
    paraphrase = back_translate(text)
    print(f'ORIGINAL    : {text}')
    print(f'BACK-TRANS  : {paraphrase}')
    print()

Loading translation pipelines...


KeyError: "Unknown task translation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

## 8. Composing Multiple Augmenters with nlpaug Pipeline

`nafl.Sequential` chains augmenters in order — all are applied.
`nafl.Sometimes` applies each augmenter with a given probability.

This lets you build a production augmentation pipeline with
fine-grained control over which techniques are applied and how often.


In [ ]:
# Build a composed augmentation pipeline
# Each augmenter applied with its own probability
composed_aug = nafl.Sequential([
    nafl.Sometimes(
        naw.SynonymAug(aug_src='wordnet', aug_p=0.15),
        pipeline_p=0.6    # apply synonym aug 60% of the time
    ),
    nafl.Sometimes(
        naw.RandomWordAug(action='delete', aug_p=0.1),
        pipeline_p=0.3    # apply deletion 30% of the time
    ),
    nafl.Sometimes(
        naw.SpellingAug(aug_p=0.1),
        pipeline_p=0.2    # apply spelling errors 20% of the time
    ),
])

print('=== Composed Pipeline (Synonym + Delete + Spelling) ===')
print()
for i, text in enumerate(DEMO_TEXTS):
    result = composed_aug.augment(text)
    if isinstance(result, list):
        result = result[0]
    print(f'--- Example {i+1} ---')
    print(f'ORIGINAL : {text[:200]}')
    print(f'AUGMENTED: {result[:200]}')
    print()

## 9. Building an Augmented Training Set

For the classifier A/B test, we augment the training set offline
(before training) rather than on-the-fly. This is faster for small
datasets and lets us inspect the augmented samples.

Strategy: for each original sample, generate 1 augmented copy.
This doubles the training set size.


In [ ]:
# Use the fast augmenters (no BERT) for dataset augmentation
fast_aug = nafl.Sequential([
    nafl.Sometimes(
        naw.SynonymAug(aug_src='wordnet', aug_p=0.15),
        pipeline_p=0.7
    ),
    nafl.Sometimes(
        naw.RandomWordAug(action='delete', aug_p=0.1),
        pipeline_p=0.3
    ),
    nafl.Sometimes(
        naw.SpellingAug(aug_p=0.08),
        pipeline_p=0.2
    ),
])


def augment_dataset(texts, labels, augmenter, n_aug=1):
    """
    Augment each sample n_aug times and append to original dataset.
    Returns combined (texts, labels) with original + augmented.
    """
    aug_texts  = list(texts)
    aug_labels = list(labels)

    print(f'Augmenting {len(texts)} samples x {n_aug} copies...')
    for i, (text, label) in enumerate(zip(texts, labels)):
        for _ in range(n_aug):
            result = augmenter.augment(text)
            aug_text = result[0] if isinstance(result, list) else result
            aug_texts.append(aug_text)
            aug_labels.append(label)

        if (i + 1) % 500 == 0:
            print(f'  Processed {i+1}/{len(texts)} samples...')

    print(f'Dataset size: {len(texts)} -> {len(aug_texts)}')
    return aug_texts, aug_labels


aug_train_texts, aug_train_labels = augment_dataset(
    train_texts, train_labels, fast_aug, n_aug=1
)

print(f'Original train size : {len(train_texts)}')
print(f'Augmented train size: {len(aug_train_texts)}')
print(f'Label distribution  : {Counter(aug_train_labels)}')

## 10. Text Classifier — Bag of Words + LSTM


In [ ]:
# Build vocabulary from training texts
def build_vocab(texts, max_vocab=10000, min_freq=2):
    counter = Counter()
    for text in texts:
        tokens = re.findall(r'[a-z]+', text.lower())
        counter.update(tokens)

    # Keep top max_vocab words with at least min_freq occurrences
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in counter.most_common(max_vocab):
        if count >= min_freq:
            vocab[word] = len(vocab)

    return vocab


def tokenize(text, vocab, max_len=128):
    tokens = re.findall(r'[a-z]+', text.lower())
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens[:max_len]]
    # Pad or truncate to max_len
    ids = ids + [vocab['<PAD>']] * max(0, max_len - len(ids))
    return ids[:max_len]


vocab = build_vocab(train_texts + test_texts)
print(f'Vocabulary size: {len(vocab)}')


class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.ids    = [tokenize(t, vocab, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.ids[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )


BATCH_SIZE = 64

# Baseline: original training data only
train_ds_base = TextDataset(train_texts, train_labels, vocab)

# Augmented: original + augmented training data
train_ds_aug  = TextDataset(aug_train_texts, aug_train_labels, vocab)

test_ds = TextDataset(test_texts, test_labels, vocab)

train_loader_base = DataLoader(train_ds_base, batch_size=BATCH_SIZE, shuffle=True)
train_loader_aug  = DataLoader(train_ds_aug,  batch_size=BATCH_SIZE, shuffle=True)
test_loader       = DataLoader(test_ds,        batch_size=BATCH_SIZE, shuffle=False)

print(f'Base train batches: {len(train_loader_base)}')
print(f'Aug  train batches: {len(train_loader_aug)}')

In [ ]:
class TextClassifier(nn.Module):
    """
    Bidirectional LSTM text classifier.

    Architecture:
    Embedding -> BiLSTM -> Mean pooling -> Dropout -> Linear -> 2 classes

    Bidirectional: reads text forward AND backward — captures context
    from both directions. Better than unidirectional for classification.
    """
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128,
                 n_layers=2, dropout=0.4, n_classes=2):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,   # forward + backward
            dropout=dropout if n_layers > 1 else 0
        )

        self.dropout    = nn.Dropout(dropout)
        # hidden_dim * 2 because bidirectional concatenates forward + backward
        self.classifier = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        output, _ = self.lstm(embedded)              # (batch, seq_len, hidden*2)
        pooled = output.mean(dim=1)                  # mean over time steps
        return self.classifier(self.dropout(pooled))


def make_model():
    return TextClassifier(
        vocab_size=len(vocab),
        embed_dim=64,
        hidden_dim=128,
        n_layers=2,
        dropout=0.4
    ).to(device)

test_m = make_model()
dummy  = torch.zeros(4, 128, dtype=torch.long).to(device)
print(f'Output shape: {test_m(dummy).shape}  (expect [4, 2])')
print(f'Parameters: {sum(p.numel() for p in test_m.parameters()):,}')

## 11. A/B Test: Baseline vs Augmented Training


In [ ]:
def train_text_model(train_loader, label='', epochs=15):
    torch.manual_seed(42)
    model     = make_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=0.5, patience=3
    )
    criterion = nn.CrossEntropyLoss()

    best_test_acc = 0.0
    hist = {'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        correct = total = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out  = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
            optimizer.step()
            correct += (out.detach().argmax(1) == y_b).sum().item()
            total   += y_b.size(0)
        train_acc = correct / total

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                correct += (model(X_b).argmax(1) == y_b).sum().item()
                total   += y_b.size(0)
        test_acc = correct / total
        best_test_acc = max(best_test_acc, test_acc)

        scheduler.step(1 - test_acc)
        hist['train_acc'].append(train_acc)
        hist['test_acc'].append(test_acc)

        if (epoch + 1) % 3 == 0:
            print(f'[{label}] Epoch {epoch+1:2d} | '
                  f'train={train_acc:.4f} | test={test_acc:.4f}')

    print(f'[{label}] Best test accuracy: {best_test_acc:.4f}\n')
    return hist, best_test_acc


print('Training baseline model (original data only)...')
hist_base, acc_base = train_text_model(
    train_loader_base, label='Baseline', epochs=15
)

print('Training augmented model (original + augmented data)...')
hist_aug, acc_aug = train_text_model(
    train_loader_aug, label='Augmented', epochs=15
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist_base['train_acc'], '--', color='tomato',    label='Baseline — train')
axes[0].plot(hist_base['test_acc'],        color='tomato',    label='Baseline — test')
axes[0].plot(hist_aug['train_acc'],  '--', color='steelblue', label='Augmented — train')
axes[0].plot(hist_aug['test_acc'],         color='steelblue', label='Augmented — test')
axes[0].set_title('Accuracy curves — IMDb sentiment')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

bars = axes[1].bar(
    ['Baseline\n(original only)', 'Augmented\n(original + aug)'],
    [acc_base, acc_aug],
    color=['tomato', 'steelblue'], width=0.4
)
for bar, acc in zip(bars, [acc_base, acc_aug]):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.005,
        f'{acc:.4f}', ha='center', fontsize=12
    )
axes[1].set_ylim(0.5, 1.0)
axes[1].set_ylabel('Best Test Accuracy')
axes[1].set_title('A/B Test Result')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'NLP Augmentation A/B Test — improvement: '
    f'{(acc_aug - acc_base)*100:.2f} pp',
    fontsize=12
)
plt.tight_layout()
plt.show()

print(f'Baseline  test accuracy: {acc_base:.4f}')
print(f'Augmented test accuracy: {acc_aug:.4f}')
print(f'Improvement: {(acc_aug-acc_base)*100:.2f} percentage points')

---
## Summary

### nlpaug Quick Reference

| Augmenter | Level | Best for |
|---|---|---|
| `nac.KeyboardAug` | Character | Noisy user input, social media |
| `nac.OcrAug` | Character | OCR post-processing robustness |
| `nac.RandomCharAug` | Character | General noise robustness |
| `naw.SynonymAug` | Word | Vocabulary diversity, paraphrasing |
| `naw.RandomWordAug(delete)` | Word | Missing word robustness |
| `naw.RandomWordAug(swap)` | Word | Word order robustness |
| `naw.SpellingAug` | Word | Noisy user input |
| `naw.ContextualWordEmbsAug` | Word | High-quality context-aware augmentation |
| Back-translation | Sentence | Best quality paraphrases, slowest |
| `nafl.Sequential` | Pipeline | Chain multiple augmenters |
| `nafl.Sometimes` | Pipeline | Apply each augmenter probabilistically |

### When to Use Each Technique
- **Low data regime** (< 1000 samples): back-translation + BERT augmentation
- **Noisy input domain** (social media, SMS): keyboard + spelling augmentation
- **Fast augmentation** (large datasets): synonym + random delete/swap
- **Highest quality**: BERT contextual augmentation

### Key Difference from Image Augmentation
Text labels must be preserved. Never augment sentiment-bearing words
with their antonyms. Keep `aug_p` low (0.1-0.2) to avoid changing meaning.

**Next: Notebook 12** — Time Series Augmentation
